In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results"

In [4]:
regime_features = pd.read_csv(
    RESULTS_DIR / "regime_features_2025.csv",
    index_col=0,
    parse_dates=True
)

monthly_analysis = pd.read_csv(
    RESULTS_DIR / "monthly_regime_strategy_analysis_2025.csv",
    index_col=0
)

eiie_concentration_summary = pd.read_csv(
    RESULTS_DIR / "eiie_concentration_summary.csv"
)

display(regime_features.head())

display(monthly_analysis)

display(eiie_concentration_summary)

,Breadth_20D,Momentum_20D,Dispersion_20D
date,,,
2025-04-16,0.45,-0.088275,0.019067
2025-04-17,0.45,-0.096330,0.018548
2025-04-18,0.45,-0.092261,0.018449
2025-04-21,0.47,-0.088425,0.017408
2025-04-22,0.46,-0.088465,0.017421


,Breadth_20D,Momentum_20D,Dispersion_20D,Static MVO,Rolling MVO,Equal Weight,EIIE 40ep,NAVER Only,Winner,Winner Return (%)
2025-04,0.471818,-0.058412,0.017376,-4.255535,-0.022066,0.925554,4.250478,4.973826,NAVER Only,4.973826
2025-05,0.523684,0.073211,0.015165,2.852853,13.954122,4.838804,-5.824894,-6.483807,Rolling MVO,13.954122
2025-06,0.572105,0.152187,0.018870,12.350584,6.847919,20.800284,37.054724,39.999989,NAVER Only,39.999989
2025-07,0.533043,0.085824,0.024159,12.978674,-6.063922,1.399298,-7.996439,-10.476197,Static MVO,12.978674
2025-08,0.503000,-0.011723,0.018423,-1.066469,-4.945377,-1.912635,-7.417444,-8.723370,Static MVO,-1.066469
2025-09,0.593810,0.081579,0.016604,17.973397,21.121668,15.707883,22.679824,25.174811,NAVER Only,25.174811
2025-10,0.573333,0.193840,0.024320,33.939450,23.440032,23.405333,-0.099516,-0.372445,Static MVO,33.939450
2025-11,0.529500,0.109388,0.024206,-6.574137,-7.175859,-4.092022,-7.395266,-8.785043,Equal Weight,-4.092022
2025-12,0.466667,0.027111,0.020100,19.399951,19.059271,10.946745,-0.440232,-0.614773,Static MVO,19.399951


,Metric,Value
0,EIIE_NAVER_Correlation,0.998752
1,Average_HHI,0.772454
2,Effective_Number_of_Assets,1.300079
3,Average_Max_Weight,0.871604


In [5]:
stock_prices_2025 = pd.read_csv(
    RESULTS_DIR / "stock_prices_2025.csv",
    index_col=0,
    parse_dates=True
)

stock_prices_2025.head()

,000660.KS,005380.KS,005930.KS,035420.KS,105560.KS
date,,,,,
2025-03-19,204231.625000,194257.859375,57083.609375,205837.796875,77372.546875
2025-03-20,208703.828125,192829.468750,58742.453125,205837.796875,78420.695312
2025-03-21,214169.875000,195210.109375,60206.136719,207322.203125,77467.835938
2025-03-24,210194.578125,202828.062500,59035.191406,204848.187500,77944.265625
2025-03-25,206716.187500,209493.765625,58352.140625,205342.984375,78039.562500


In [6]:
#종목별 20일 momentum
#20거래일 누적수익률

stock_momentum_20d = (
    stock_prices_2025
    / stock_prices_2025.shift(20)
    - 1
)

stock_momentum_20d.head(25)


,000660.KS,005380.KS,005930.KS,035420.KS,105560.KS
date,,,,,
2025-03-19,NaN,NaN,NaN,NaN,NaN
2025-03-20,NaN,NaN,NaN,NaN,NaN
2025-03-21,NaN,NaN,NaN,NaN,NaN
2025-03-24,NaN,NaN,NaN,NaN,NaN
2025-03-25,NaN,NaN,NaN,NaN,NaN
2025-03-26,NaN,NaN,NaN,NaN,NaN
2025-03-27,NaN,NaN,NaN,NaN,NaN
2025-03-28,NaN,NaN,NaN,NaN,NaN
2025-03-31,NaN,NaN,NaN,NaN,NaN


In [7]:
stock_momentum_20d.loc["2025-10-01"]

000660.KS    0.381958
005380.KS   -0.020455
005930.KS    0.250126
035420.KS    0.131111
105560.KS    0.068934
Name: 2025-10-01 00:00:00, dtype: float64

In [9]:
# 20일 Momentum이 계산된 날짜만 사용
valid_stock_momentum_20d = (
    stock_momentum_20d
    .dropna(how="all")
)

# 가장 강한 종목
leader_ticker = (
    valid_stock_momentum_20d
    .idxmax(axis=1)
)

# 가장 높은 20일 Momentum
leader_return = (
    valid_stock_momentum_20d
    .max(axis=1)
)

leadership = pd.DataFrame({
    "Leader": leader_ticker,
    "Leader_Momentum_20D": leader_return
})

leadership.head()

,Leader,Leader_Momentum_20D
date,,
2025-04-16,105560.KS,-0.014778
2025-04-17,105560.KS,-0.024301
2025-04-18,105560.KS,0.013530
2025-04-21,105560.KS,0.012225
2025-04-22,105560.KS,0.018315


In [10]:
#leadership gap

sorted_momentum = np.sort(
    valid_stock_momentum_20d.values,
    axis=1
)

leadership_gap = pd.Series(
    sorted_momentum[:, -1]
    - sorted_momentum[:, -2],
    index=valid_stock_momentum_20d.index,
    name="Leadership_Gap_20D"
)

leadership_gap.head()

date
2025-04-16    0.044624
2025-04-17    0.054978
2025-04-18    0.100359
2025-04-21    0.091082
2025-04-22    0.084821
Name: Leadership_Gap_20D, dtype: float64

In [16]:
#semiconductor leadership

semiconductor_momentum_20d = (
    stock_momentum_20d[
        [
            "000660.KS",
            "005930.KS"
        ]
    ]
    .mean(axis=1)
)

naver_momentum_20d = (
    stock_momentum_20d[
        "035420.KS"
    ]
)

auto_momentum_20d = (
    stock_momentum_20d[
        "005380.KS"
    ]
)

finance_momentum_20d = (
    stock_momentum_20d[
        "105560.KS"
    ]
)

In [17]:
#momemtum 변수 기준 통일

semiconductor_momentum_20d = (
    valid_stock_momentum_20d[
        ["000660.KS", "005930.KS"]
    ]
    .mean(axis=1)
)

naver_momentum_20d = (
    valid_stock_momentum_20d["035420.KS"]
)

auto_momentum_20d = (
    valid_stock_momentum_20d["005380.KS"]
)

finance_momentum_20d = (
    valid_stock_momentum_20d["105560.KS"]
)

universe_momentum_20d = (
    valid_stock_momentum_20d
    .mean(axis=1)
)

In [18]:
#상대강도
#Relative Strength = 종목 Momentum - 전체 Universe Momentum

naver_relative_strength = (
    naver_momentum_20d
    - universe_momentum_20d
)

semiconductor_relative_strength = (
    semiconductor_momentum_20d
    - universe_momentum_20d
)

auto_relative_strength = (
    auto_momentum_20d
    - universe_momentum_20d
)

finance_relative_strength = (
    finance_momentum_20d
    - universe_momentum_20d
)

In [14]:
#leadership feature 묶

leadership_features = pd.DataFrame({
    "Leader": leader_ticker,
    "Leader_Momentum_20D": leader_return,
    "Leadership_Gap_20D": leadership_gap,

    "NAVER_Momentum_20D":
        naver_momentum_20d,

    "Semiconductor_Momentum_20D":
        semiconductor_momentum_20d,

    "Auto_Momentum_20D":
        auto_momentum_20d,

    "Finance_Momentum_20D":
        finance_momentum_20d,

    "NAVER_Relative_Strength":
        naver_relative_strength,

    "Semiconductor_Relative_Strength":
        semiconductor_relative_strength,

    "Auto_Relative_Strength":
        auto_relative_strength,

    "Finance_Relative_Strength":
        finance_relative_strength
})

leadership_features.head()

,Leader,Leader_Momentum_20D,Leadership_Gap_20D,NAVER_Momentum_20D,Semiconductor_Momentum_20D,Auto_Momentum_20D,Finance_Momentum_20D,NAVER_Relative_Strength,Semiconductor_Relative_Strength,Auto_Relative_Strength,Finance_Relative_Strength
date,,,,,,,,,,,
2025-04-16,105560.KS,-0.014778,0.044624,-0.113462,-0.106343,-0.109314,-0.014778,-0.023414,-0.016295,-0.019266,0.075270
2025-04-17,105560.KS,-0.024301,0.054978,-0.117308,-0.122973,-0.102716,-0.024301,-0.019253,-0.024919,-0.004662,0.073753
2025-04-18,105560.KS,0.013530,0.100359,-0.105012,-0.143169,-0.086829,0.013530,-0.012082,-0.050239,0.006101,0.106460
2025-04-21,105560.KS,0.012225,0.091082,-0.094203,-0.121935,-0.120657,0.012225,-0.004902,-0.032634,-0.031356,0.101526
2025-04-22,105560.KS,0.018315,0.084821,-0.066506,-0.119613,-0.156364,0.018315,0.022250,-0.030857,-0.067607,0.107071


In [19]:
#기존 regime feature와 합치기

market_state_2025 = pd.concat(
    [
        regime_features,
        leadership_features
    ],
    axis=1
)

market_state_2025 = (
    market_state_2025
    .dropna()
)

market_state_2025.head()

,Breadth_20D,Momentum_20D,Dispersion_20D,Leader,Leader_Momentum_20D,Leadership_Gap_20D,NAVER_Momentum_20D,Semiconductor_Momentum_20D,Auto_Momentum_20D,Finance_Momentum_20D,NAVER_Relative_Strength,Semiconductor_Relative_Strength,Auto_Relative_Strength,Finance_Relative_Strength
date,,,,,,,,,,,,,,
2025-04-16,0.45,-0.088275,0.019067,105560.KS,-0.014778,0.044624,-0.113462,-0.106343,-0.109314,-0.014778,-0.023414,-0.016295,-0.019266,0.075270
2025-04-17,0.45,-0.096330,0.018548,105560.KS,-0.024301,0.054978,-0.117308,-0.122973,-0.102716,-0.024301,-0.019253,-0.024919,-0.004662,0.073753
2025-04-18,0.45,-0.092261,0.018449,105560.KS,0.013530,0.100359,-0.105012,-0.143169,-0.086829,0.013530,-0.012082,-0.050239,0.006101,0.106460
2025-04-21,0.47,-0.088425,0.017408,105560.KS,0.012225,0.091082,-0.094203,-0.121935,-0.120657,0.012225,-0.004902,-0.032634,-0.031356,0.101526
2025-04-22,0.46,-0.088465,0.017421,105560.KS,0.018315,0.084821,-0.066506,-0.119613,-0.156364,0.018315,0.022250,-0.030857,-0.067607,0.107071


In [20]:
display(
    market_state_2025.loc[
        "2025-06-02":"2025-06-30"
    ].tail()
)

display(
    market_state_2025.loc[
        "2025-10-01":"2025-10-31"
    ].tail()
)

,Breadth_20D,Momentum_20D,Dispersion_20D,Leader,Leader_Momentum_20D,Leadership_Gap_20D,NAVER_Momentum_20D,Semiconductor_Momentum_20D,Auto_Momentum_20D,Finance_Momentum_20D,NAVER_Relative_Strength,Semiconductor_Relative_Strength,Auto_Relative_Strength,Finance_Relative_Strength
date,,,,,,,,,,,,,,
2025-06-24,0.65,0.273365,0.022785,035420.KS,0.586565,0.191550,0.586565,0.255625,0.160690,0.134343,0.307995,-0.022944,-0.117880,-0.144226
2025-06-25,0.63,0.261391,0.023998,035420.KS,0.507979,0.096567,0.507979,0.266035,0.209225,0.073529,0.243418,0.001474,-0.055335,-0.191031
2025-06-26,0.65,0.245283,0.025482,000660.KS,0.449527,0.029505,0.420022,0.283205,0.167432,0.078508,0.173547,0.036731,-0.079043,-0.167966
2025-06-27,0.63,0.210670,0.025597,035420.KS,0.371870,0.004019,0.371870,0.231089,0.117775,0.095049,0.162495,0.021715,-0.091599,-0.114325
2025-06-30,0.61,0.195235,0.025787,035420.KS,0.386688,0.009329,0.386688,0.224925,0.065445,0.079844,0.190322,0.028560,-0.130920,-0.116521


,Breadth_20D,Momentum_20D,Dispersion_20D,Leader,Leader_Momentum_20D,Leadership_Gap_20D,NAVER_Momentum_20D,Semiconductor_Momentum_20D,Auto_Momentum_20D,Finance_Momentum_20D,NAVER_Relative_Strength,Semiconductor_Relative_Strength,Auto_Relative_Strength,Finance_Relative_Strength
date,,,,,,,,,,,,,,
2025-10-27,0.57,0.196632,0.024023,000660.KS,0.524217,0.297210,0.081897,0.375612,0.167431,0.009410,-0.120096,0.173620,-0.034561,-0.192582
2025-10-28,0.55,0.172298,0.023899,000660.KS,0.443213,0.263238,0.095238,0.311594,0.143836,0.008666,-0.078948,0.137409,-0.030350,-0.165520
2025-10-29,0.58,0.214014,0.024958,000660.KS,0.560839,0.374632,0.162281,0.371452,0.186207,0.006071,-0.057212,0.151960,-0.033286,-0.213422
2025-10-30,0.59,0.198529,0.023785,000660.KS,0.593268,0.372070,0.005906,0.403861,0.221198,0.013123,-0.203684,0.194271,0.011608,-0.196466
2025-10-31,0.62,0.264464,0.024642,000660.KS,0.661218,0.309237,0.042885,0.478746,0.351981,0.034605,-0.234508,0.201353,0.074589,-0.242788


In [21]:
#월평균

monthly_market_state = (
    market_state_2025
    .drop(columns=["Leader"])
    .resample("ME")
    .mean()
)

monthly_market_state.index = (
    monthly_market_state
    .index
    .strftime("%Y-%m")
)

monthly_market_state.round(3)

,Breadth_20D,Momentum_20D,Dispersion_20D,Leader_Momentum_20D,Leadership_Gap_20D,NAVER_Momentum_20D,Semiconductor_Momentum_20D,Auto_Momentum_20D,Finance_Momentum_20D,NAVER_Relative_Strength,Semiconductor_Relative_Strength,Auto_Relative_Strength,Finance_Relative_Strength
date,,,,,,,,,,,,,
2025-04,0.472,-0.058,0.017,0.048,0.085,-0.049,-0.101,-0.092,0.048,0.010,-0.042,-0.033,0.107
2025-05,0.524,0.073,0.015,0.225,0.101,-0.000,0.064,0.020,0.225,-0.075,-0.011,-0.055,0.151
2025-06,0.572,0.152,0.019,0.315,0.073,0.226,0.165,0.080,0.129,0.073,0.012,-0.073,-0.024
2025-07,0.533,0.086,0.024,0.214,0.067,0.087,0.108,0.054,0.062,0.003,0.024,-0.030,-0.021
2025-08,0.503,-0.012,0.018,0.085,0.071,-0.076,0.013,0.018,-0.034,-0.063,0.027,0.031,-0.021
2025-09,0.594,0.082,0.017,0.221,0.110,0.056,0.153,0.019,0.034,-0.027,0.070,-0.063,-0.049
2025-10,0.573,0.194,0.024,0.496,0.215,0.100,0.387,0.116,0.011,-0.100,0.187,-0.084,-0.189
2025-11,0.530,0.109,0.024,0.305,0.170,0.007,0.167,0.122,0.088,-0.103,0.057,0.012,-0.022
2025-12,0.467,0.027,0.020,0.124,0.044,-0.070,0.044,0.095,0.016,-0.095,0.018,0.069,-0.010


In [22]:
#저장
market_state_2025.to_csv(
    RESULTS_DIR / "market_state_features_2025.csv",
    encoding="utf-8-sig"
)

monthly_market_state.to_csv(
    RESULTS_DIR / "monthly_market_state_2025.csv",
    encoding="utf-8-sig"
)

print("Leadership / Market State 저장 완료")

Leadership / Market State 저장 완료


In [23]:
# 현재 주도 그룹 자동 판정

relative_strength_cols = {
    "NAVER": "NAVER_Relative_Strength",
    "Semiconductor": "Semiconductor_Relative_Strength",
    "Auto": "Auto_Relative_Strength",
    "Finance": "Finance_Relative_Strength"
}

relative_strength_df = (
    market_state_2025[
        list(relative_strength_cols.values())
    ]
    .rename(
        columns={
            v: k
            for k, v in relative_strength_cols.items()
        }
    )
)

market_state_2025["Leadership_Group"] = (
    relative_strength_df.idxmax(axis=1)
)

In [24]:
market_state_2025[
    [
        "Leadership_Group",
        "NAVER_Relative_Strength",
        "Semiconductor_Relative_Strength",
        "Auto_Relative_Strength",
        "Finance_Relative_Strength"
    ]
].head()

,Leadership_Group,NAVER_Relative_Strength,Semiconductor_Relative_Strength,Auto_Relative_Strength,Finance_Relative_Strength
date,,,,,
2025-04-16,Finance,-0.023414,-0.016295,-0.019266,0.075270
2025-04-17,Finance,-0.019253,-0.024919,-0.004662,0.073753
2025-04-18,Finance,-0.012082,-0.050239,0.006101,0.106460
2025-04-21,Finance,-0.004902,-0.032634,-0.031356,0.101526
2025-04-22,Finance,0.022250,-0.030857,-0.067607,0.107071


In [26]:
#상승/하락 regime
#bull:상승, bear:하락

market_state_2025["Market_Direction"] = np.where(
    market_state_2025["Momentum_20D"] > 0,
    "Bull",
    "Bear"
)

In [27]:
#broad: 최근 20일 동안 비교적 많은 종목이 함께 상승
#narrow: 상승이 일부 종목에 상대적으로 집중

BREADTH_THRESHOLD = 0.55

market_state_2025["Breadth_Regime"] = np.where(
    market_state_2025["Breadth_20D"] >= BREADTH_THRESHOLD,
    "Broad",
    "Narrow"
)

In [28]:
#leadership이 강한지도 구분

LEADERSHIP_GAP_THRESHOLD = (
    market_state_2025[
        "Leadership_Gap_20D"
    ]
    .median()
)

print(
    "Leadership Gap threshold:",
    LEADERSHIP_GAP_THRESHOLD
)

Leadership Gap threshold: 0.08576714791245343


In [29]:
market_state_2025["Leadership_Strength"] = np.where(
    market_state_2025[
        "Leadership_Gap_20D"
    ] >= LEADERSHIP_GAP_THRESHOLD,
    "Strong",
    "Weak"
)

In [30]:
#regime label

market_state_2025["Regime_Label"] = (
    market_state_2025["Market_Direction"]
    + "_"
    + market_state_2025["Breadth_Regime"]
    + "_"
    + market_state_2025["Leadership_Group"]
    + "_"
    + market_state_2025["Leadership_Strength"]
)

In [31]:
#확인

market_state_2025[
    [
        "Momentum_20D",
        "Breadth_20D",
        "Leadership_Group",
        "Leadership_Strength",
        "Regime_Label"
    ]
].tail(20)

,Momentum_20D,Breadth_20D,Leadership_Group,Leadership_Strength,Regime_Label
date,,,,,
2025-12-02,-0.015129,0.43,Finance,Strong,Bear_Narrow_Finance_Strong
2025-12-03,-0.004527,0.45,Finance,Weak,Bear_Narrow_Finance_Weak
2025-12-04,0.003638,0.45,Auto,Weak,Bull_Narrow_Auto_Weak
2025-12-05,0.055184,0.50,Auto,Strong,Bull_Narrow_Auto_Strong
2025-12-08,0.036522,0.49,Auto,Strong,Bull_Narrow_Auto_Strong
2025-12-09,0.007003,0.45,Auto,Strong,Bull_Narrow_Auto_Strong
2025-12-10,-0.004331,0.43,Auto,Weak,Bear_Narrow_Auto_Weak
2025-12-11,-0.019097,0.42,Auto,Weak,Bear_Narrow_Auto_Weak
2025-12-12,0.041955,0.47,Auto,Weak,Bull_Narrow_Auto_Weak


In [32]:
#6월과 10월비교

print("=== 2025 June ===")

display(
    market_state_2025.loc[
        "2025-06-01":"2025-06-30",
        [
            "Momentum_20D",
            "Breadth_20D",
            "Leadership_Group",
            "Leadership_Gap_20D",
            "Regime_Label"
        ]
    ].tail()
)

print("=== 2025 October ===")

display(
    market_state_2025.loc[
        "2025-10-01":"2025-10-31",
        [
            "Momentum_20D",
            "Breadth_20D",
            "Leadership_Group",
            "Leadership_Gap_20D",
            "Regime_Label"
        ]
    ].tail()
)

=== 2025 June ===


,Momentum_20D,Breadth_20D,Leadership_Group,Leadership_Gap_20D,Regime_Label
date,,,,,
2025-06-24,0.273365,0.65,NAVER,0.191550,Bull_Broad_NAVER_Strong
2025-06-25,0.261391,0.63,NAVER,0.096567,Bull_Broad_NAVER_Strong
2025-06-26,0.245283,0.65,NAVER,0.029505,Bull_Broad_NAVER_Weak
2025-06-27,0.210670,0.63,NAVER,0.004019,Bull_Broad_NAVER_Weak
2025-06-30,0.195235,0.61,NAVER,0.009329,Bull_Broad_NAVER_Weak


=== 2025 October ===


,Momentum_20D,Breadth_20D,Leadership_Group,Leadership_Gap_20D,Regime_Label
date,,,,,
2025-10-27,0.196632,0.57,Semiconductor,0.297210,Bull_Broad_Semiconductor_Strong
2025-10-28,0.172298,0.55,Semiconductor,0.263238,Bull_Broad_Semiconductor_Strong
2025-10-29,0.214014,0.58,Semiconductor,0.374632,Bull_Broad_Semiconductor_Strong
2025-10-30,0.198529,0.59,Semiconductor,0.372070,Bull_Broad_Semiconductor_Strong
2025-10-31,0.264464,0.62,Semiconductor,0.309237,Bull_Broad_Semiconductor_Strong


In [ ]:
#regime 규칙
#Bear → Cash, Bull + Strong NAVER leadership → EIIE
#Bull + Strong Semiconductor leadership → Static MVO, 그 외 Bull → Equal Weight

In [33]:
MOMENTUM_THRESHOLD = 0.0
BREADTH_THRESHOLD = 0.55
LEADERSHIP_GAP_THRESHOLD = 0.08576714791245343

In [34]:
def select_strategy(row):

    # 하락장
    if row["Momentum_20D"] <= MOMENTUM_THRESHOLD:
        return "Cash"

    # 상승장 + 강한 주도주 존재
    if row["Leadership_Gap_20D"] >= LEADERSHIP_GAP_THRESHOLD:

        if row["Leadership_Group"] == "NAVER":
            return "EIIE 40ep"

        if row["Leadership_Group"] == "Semiconductor":
            return "Static MVO"

    # 그 외에는 중립적인 분산 전략
    return "Equal Weight"

In [35]:
market_state_2025["Selected_Strategy_V1"] = (
    market_state_2025.apply(
        select_strategy,
        axis=1
    )
)

market_state_2025[
    [
        "Momentum_20D",
        "Leadership_Group",
        "Leadership_Gap_20D",
        "Selected_Strategy_V1"
    ]
].tail(20)

,Momentum_20D,Leadership_Group,Leadership_Gap_20D,Selected_Strategy_V1
date,,,,
2025-12-02,-0.015129,Finance,0.104682,Cash
2025-12-03,-0.004527,Finance,0.051821,Cash
2025-12-04,0.003638,Auto,0.004601,Equal Weight
2025-12-05,0.055184,Auto,0.097447,Equal Weight
2025-12-08,0.036522,Auto,0.089148,Equal Weight
2025-12-09,0.007003,Auto,0.104937,Equal Weight
2025-12-10,-0.004331,Auto,0.061076,Cash
2025-12-11,-0.019097,Auto,0.027509,Cash
2025-12-12,0.041955,Auto,0.003269,Equal Weight


In [36]:
strategy_rule_config = pd.DataFrame({
    "Parameter": [
        "MOMENTUM_THRESHOLD",
        "BREADTH_THRESHOLD",
        "LEADERSHIP_GAP_THRESHOLD"
    ],
    "Value": [
        MOMENTUM_THRESHOLD,
        BREADTH_THRESHOLD,
        LEADERSHIP_GAP_THRESHOLD
    ]
})

strategy_rule_config.to_csv(
    RESULTS_DIR / "regime_strategy_v1_config.csv",
    index=False,
    encoding="utf-8-sig"
)

strategy_rule_config

,Parameter,Value
0,MOMENTUM_THRESHOLD,0.000000
1,BREADTH_THRESHOLD,0.550000
2,LEADERSHIP_GAP_THRESHOLD,0.085767


In [37]:
market_state_2025.to_csv(
    RESULTS_DIR / "market_state_2025_with_strategy_v1.csv",
    encoding="utf-8-sig"
)

In [39]:
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [40]:
KOREA_STOCKS = [
    "005930.KS",
    "000660.KS",
    "005380.KS",
    "035420.KS",
    "105560.KS"
]

TEST_2026_START = "2026-01-01"

# end는 일반적으로 exclusive하게 처리되므로
# 9/15까지 포함하기 위해 다음 날짜 지정
TEST_2026_END = "2026-09-16"

In [41]:
raw_2026 = YahooDownloader(
    start_date=TEST_2026_START,
    end_date=TEST_2026_END,
    ticker_list=KOREA_STOCKS
).fetch_data()

raw_2026["date"] = pd.to_datetime(
    raw_2026["date"]
)

print(raw_2026.shape)

raw_2026.head()

YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (865, 8)
(865, 8)


Price,date,close,high,low,open,volume,tic,day
0,2026-01-02,675493.625000,677489.174852,645560.377216,649551.476920,4181895,000660.KS,4
1,2026-01-02,294244.375000,297694.476549,288822.786851,295230.118300,955205,005380.KS,4
2,2026-01-02,128093.312500,128093.312500,119819.581031,119819.581031,30463279,005930.KS,4
3,2026-01-02,244432.375000,246411.584514,235525.932186,240968.758350,1362199,035420.KS,4
4,2026-01-02,120421.148438,121788.460747,119932.822613,120714.143932,648227,105560.KS,4


In [42]:
#공통 거래일 필터

date_counts_2026 = (
    raw_2026
    .groupby("date")["tic"]
    .nunique()
)

common_dates_2026 = date_counts_2026[
    date_counts_2026 == len(KOREA_STOCKS)
].index

raw_2026 = (
    raw_2026[
        raw_2026["date"].isin(common_dates_2026)
    ]
    .sort_values(["date", "tic"])
    .reset_index(drop=True)
)

print(
    raw_2026.groupby("tic").size()
)

tic
000660.KS    173
005380.KS    173
005930.KS    173
035420.KS    173
105560.KS    173
dtype: int64


In [43]:
#가격표

prices_2026 = (
    raw_2026
    .pivot(
        index="date",
        columns="tic",
        values="close"
    )
    .sort_index()
)

print(prices_2026.shape)
print(prices_2026.index.min())
print(prices_2026.index.max())

prices_2026.head()

(173, 5)
2026-01-02 00:00:00
2026-09-15 00:00:00


tic,000660.KS,005380.KS,005930.KS,035420.KS,105560.KS
date,,,,,
2026-01-02,675493.6250,294244.37500,128093.312500,244432.375000,120421.148438
2026-01-05,694451.3750,300158.87500,137662.937500,246906.390625,123839.429688
2026-01-06,724384.6250,303608.93750,138460.390625,257297.234375,123448.765625
2026-01-07,740348.9375,345503.03125,140553.765625,249875.203125,121788.468750
2026-01-08,754317.8750,335645.62500,138360.703125,248390.796875,120616.484375


In [44]:
#2025-2026 가격 연결

stock_prices_2025 = pd.read_csv(
    RESULTS_DIR / "stock_prices_2025.csv",
    index_col=0,
    parse_dates=True
)

prices_for_2026_features = pd.concat(
    [
        stock_prices_2025,
        prices_2026
    ]
)

prices_for_2026_features = (
    prices_for_2026_features
    [~prices_for_2026_features.index.duplicated(keep="last")]
    .sort_index()
)

print(prices_for_2026_features.index.min())
print(prices_for_2026_features.index.max())
print(prices_for_2026_features.shape)

2025-03-19 00:00:00
2026-09-15 00:00:00
(365, 5)


In [45]:
#2026 일간수익률

stock_daily_returns_all = (
    prices_for_2026_features
    .pct_change()
)

In [46]:
#breadth

market_breadth_all = (
    (stock_daily_returns_all > 0)
    .sum(axis=1)
    / len(KOREA_STOCKS)
)

market_breadth_all.loc[
    stock_daily_returns_all.isna().all(axis=1)
] = np.nan

breadth_20d_all = (
    market_breadth_all
    .rolling(20)
    .mean()
)

In [47]:
#universe momentum

universe_daily_return_all = (
    stock_daily_returns_all
    .mean(axis=1)
)

momentum_20d_all = (
    (1 + universe_daily_return_all)
    .rolling(20)
    .apply(np.prod, raw=True)
    - 1
)

In [48]:
#dispersion

dispersion_20d_all = (
    stock_daily_returns_all
    .std(axis=1)
    .rolling(20)
    .mean()
)

In [49]:
#종목별 20일 momentum

stock_momentum_20d_all = (
    prices_for_2026_features
    / prices_for_2026_features.shift(20)
    - 1
)

valid_stock_momentum_20d_all = (
    stock_momentum_20d_all
    .dropna(how="all")
)

In [50]:
#leadership

leader_ticker_all = (
    valid_stock_momentum_20d_all
    .idxmax(axis=1)
)

leader_return_all = (
    valid_stock_momentum_20d_all
    .max(axis=1)
)

In [51]:
#leadership gap

sorted_momentum_all = np.sort(
    valid_stock_momentum_20d_all.values,
    axis=1
)

leadership_gap_all = pd.Series(
    sorted_momentum_all[:, -1]
    - sorted_momentum_all[:, -2],
    index=valid_stock_momentum_20d_all.index,
    name="Leadership_Gap_20D"
)

In [52]:
#그룹별 momentum

semiconductor_momentum_20d_all = (
    valid_stock_momentum_20d_all[
        ["000660.KS", "005930.KS"]
    ]
    .mean(axis=1)
)

naver_momentum_20d_all = (
    valid_stock_momentum_20d_all[
        "035420.KS"
    ]
)

auto_momentum_20d_all = (
    valid_stock_momentum_20d_all[
        "005380.KS"
    ]
)

finance_momentum_20d_all = (
    valid_stock_momentum_20d_all[
        "105560.KS"
    ]
)

universe_momentum_stock_all = (
    valid_stock_momentum_20d_all
    .mean(axis=1)
)

In [ ]:
#momentum_20d_all → 매일 5종목 평균수익률을 만든 뒤 20일 복리

#universe_momentum_stock_all → 각 종목의 20일 수익률을 먼저 만든 뒤 5종목 평균

In [53]:
#relative strength

naver_relative_strength_all = (
    naver_momentum_20d_all
    - universe_momentum_stock_all
)

semiconductor_relative_strength_all = (
    semiconductor_momentum_20d_all
    - universe_momentum_stock_all
)

auto_relative_strength_all = (
    auto_momentum_20d_all
    - universe_momentum_stock_all
)

finance_relative_strength_all = (
    finance_momentum_20d_all
    - universe_momentum_stock_all
)

In [54]:
#하나로 묶기

market_state_all = pd.DataFrame({
    "Breadth_20D":
        breadth_20d_all,

    "Momentum_20D":
        momentum_20d_all,

    "Dispersion_20D":
        dispersion_20d_all,

    "Leader":
        leader_ticker_all,

    "Leader_Momentum_20D":
        leader_return_all,

    "Leadership_Gap_20D":
        leadership_gap_all,

    "NAVER_Relative_Strength":
        naver_relative_strength_all,

    "Semiconductor_Relative_Strength":
        semiconductor_relative_strength_all,

    "Auto_Relative_Strength":
        auto_relative_strength_all,

    "Finance_Relative_Strength":
        finance_relative_strength_all
})

market_state_all = (
    market_state_all
    .dropna()
)

In [55]:
#2026만 자름

market_state_2026 = (
    market_state_all
    .loc["2026-01-02":"2026-09-15"]
    .copy()
)

print(
    "2026 Market State:",
    market_state_2026.index.min(),
    "~",
    market_state_2026.index.max()
)

print(
    "Shape:",
    market_state_2026.shape
)

market_state_2026.head()

2026 Market State: 2026-01-02 00:00:00 ~ 2026-09-15 00:00:00
Shape: (173, 10)


,Breadth_20D,Momentum_20D,Dispersion_20D,Leader,Leader_Momentum_20D,Leadership_Gap_20D,NAVER_Relative_Strength,Semiconductor_Relative_Strength,Auto_Relative_Strength,Finance_Relative_Strength
date,,,,,,,,,,
2026-01-02,0.53,0.104511,0.018703,005930.KS,0.248788,0.035526,-0.090495,0.124068,0.015224,-0.172865
2026-01-05,0.55,0.137001,0.019442,005930.KS,0.327955,0.067086,-0.129105,0.153137,0.001313,-0.178482
2026-01-06,0.56,0.152344,0.018794,000660.KS,0.339483,0.011461,-0.104953,0.176168,-0.071165,-0.176218
2026-01-07,0.54,0.143826,0.019867,000660.KS,0.363970,0.056909,-0.137945,0.185546,-0.037271,-0.195876
2026-01-08,0.52,0.123482,0.019201,000660.KS,0.310225,0.036484,-0.119335,0.164616,-0.048128,-0.161769


In [56]:
#2025에서 고정한 leadership 판정 적용

relative_strength_cols = {
    "NAVER": "NAVER_Relative_Strength",
    "Semiconductor": "Semiconductor_Relative_Strength",
    "Auto": "Auto_Relative_Strength",
    "Finance": "Finance_Relative_Strength"
}

relative_strength_2026 = (
    market_state_2026[
        list(relative_strength_cols.values())
    ]
    .rename(
        columns={
            v: k
            for k, v in relative_strength_cols.items()
        }
    )
)

market_state_2026["Leadership_Group"] = (
    relative_strength_2026
    .idxmax(axis=1)
)

In [57]:
#기존 threshold
MOMENTUM_THRESHOLD = 0.0
BREADTH_THRESHOLD = 0.55
LEADERSHIP_GAP_THRESHOLD = 0.08576714791245343

In [58]:
#전략 결정용 데이터

signal_2026 = (
    market_state_2026
    .shift(1)
)

In [59]:
#기존 v1 함수

def select_strategy(row):

    if pd.isna(row["Momentum_20D"]):
        return np.nan

    if row["Momentum_20D"] <= MOMENTUM_THRESHOLD:
        return "Cash"

    if (
        row["Leadership_Gap_20D"]
        >= LEADERSHIP_GAP_THRESHOLD
    ):

        if row["Leadership_Group"] == "NAVER":
            return "EIIE 40ep"

        if row["Leadership_Group"] == "Semiconductor":
            return "Static MVO"

    return "Equal Weight"

In [60]:
#적용

market_state_2026["Selected_Strategy_V1"] = (
    signal_2026
    .apply(
        select_strategy,
        axis=1
    )
)

In [61]:
#확인

market_state_2026[
    [
        "Momentum_20D",
        "Breadth_20D",
        "Leadership_Group",
        "Leadership_Gap_20D",
        "Selected_Strategy_V1"
    ]
].head(15)

,Momentum_20D,Breadth_20D,Leadership_Group,Leadership_Gap_20D,Selected_Strategy_V1
date,,,,,
2026-01-02,0.104511,0.53,Semiconductor,0.035526,NaN
2026-01-05,0.137001,0.55,Semiconductor,0.067086,Equal Weight
2026-01-06,0.152344,0.56,Semiconductor,0.011461,Equal Weight
2026-01-07,0.143826,0.54,Semiconductor,0.056909,Equal Weight
2026-01-08,0.123482,0.52,Semiconductor,0.036484,Equal Weight
2026-01-09,0.162817,0.56,Semiconductor,0.025966,Equal Weight
2026-01-12,0.165727,0.58,Semiconductor,0.015453,Equal Weight
2026-01-13,0.215570,0.60,Auto,0.067748,Equal Weight
2026-01-14,0.206547,0.59,Auto,0.065368,Equal Weight


In [62]:
market_state_2026[
    "Selected_Strategy_V1"
].value_counts()

Selected_Strategy_V1
Equal Weight    79
Cash            52
Static MVO      38
EIIE 40ep        3
Name: count, dtype: int64

In [63]:
#EIIE가 선택됐는지 확인

market_state_2026[
    market_state_2026["Selected_Strategy_V1"]
    == "EIIE 40ep"
][
    [
        "Momentum_20D",
        "Leadership_Group",
        "Leadership_Gap_20D",
        "Selected_Strategy_V1"
    ]
]

,Momentum_20D,Leadership_Group,Leadership_Gap_20D,Selected_Strategy_V1
date,,,,
2026-06-09,0.155271,Semiconductor,0.114725,EIIE 40ep
2026-06-15,0.106376,NAVER,0.002704,EIIE 40ep
2026-08-18,0.072382,NAVER,0.066884,EIIE 40ep


In [64]:
#static mvo

market_state_2026[
    market_state_2026["Selected_Strategy_V1"]
    == "Static MVO"
][
    [
        "Momentum_20D",
        "Leadership_Group",
        "Leadership_Gap_20D"
    ]
].head(10)

,Momentum_20D,Leadership_Group,Leadership_Gap_20D
date,,,
2026-03-09,-0.026536,Finance,0.001241
2026-04-21,0.121672,Semiconductor,0.084535
2026-04-27,0.191054,Semiconductor,0.206556
2026-04-28,0.270087,Semiconductor,0.283153
2026-04-29,0.176105,Semiconductor,0.249803
2026-04-30,0.212058,Semiconductor,0.313411
2026-05-04,0.225231,Semiconductor,0.403169
2026-05-06,0.277546,Semiconductor,0.429473
2026-05-07,0.294224,Semiconductor,0.423998


In [66]:
stock_returns_2026 = (
    prices_2026
    .pct_change()
)

stock_returns_2026.head()

tic,000660.KS,005380.KS,005930.KS,035420.KS,105560.KS
date,,,,,
2026-01-02,NaN,NaN,NaN,NaN,NaN
2026-01-05,0.028065,0.020101,0.074708,0.010121,0.028386
2026-01-06,0.043103,0.011494,0.005793,0.042084,-0.003155
2026-01-07,0.022038,0.137987,0.015119,-0.028846,-0.013449
2026-01-08,0.018868,-0.028531,-0.015603,-0.005941,-0.009623


In [67]:
print(prices_2026.columns.tolist())

['000660.KS', '005380.KS', '005930.KS', '035420.KS', '105560.KS']


In [68]:
STATIC_MVO_WEIGHTS = pd.Series(
    {
        "000660.KS": 0.14865851,
        "005380.KS": 0.16716833,
        "005930.KS": 0.68417316,
        "035420.KS": 0.0,
        "105560.KS": 0.0
    }
)

In [69]:
static_mvo_return_2026 = (
    stock_returns_2026
    .mul(
        STATIC_MVO_WEIGHTS,
        axis=1
    )
    .sum(axis=1)
)

In [70]:
#equal weight

EQUAL_WEIGHTS = pd.Series(
    0.2,
    index=prices_2026.columns
)

equal_weight_return_2026 = (
    stock_returns_2026
    .mul(
        EQUAL_WEIGHTS,
        axis=1
    )
    .sum(axis=1)
)

In [71]:
cash_return_2026 = pd.Series(
    0.0,
    index=stock_returns_2026.index
)

In [72]:
baseline_returns_2026 = pd.DataFrame({
    "Static MVO": static_mvo_return_2026,
    "Equal Weight": equal_weight_return_2026,
    "Cash": cash_return_2026
})

baseline_returns_2026.head()

,Static MVO,Equal Weight,Cash
date,,,
2026-01-02,0.000000,0.000000,0.0
2026-01-05,0.058646,0.032276,0.0
2026-01-06,0.012292,0.019864,0.0
2026-01-07,0.036687,0.026570,0.0
2026-01-08,-0.012640,-0.008166,0.0


In [73]:
baseline_cumulative_2026 = (
    (1 + baseline_returns_2026)
    .cumprod()
)

baseline_cumulative_2026.tail()

,Static MVO,Equal Weight,Cash
date,,,
2026-09-09,2.098321,1.700536,1.0
2026-09-10,2.096057,1.699761,1.0
2026-09-11,2.032662,1.680923,1.0
2026-09-14,1.947442,1.639238,1.0
2026-09-15,1.939629,1.626000,1.0


In [74]:
baseline_total_returns_2026 = (
    baseline_cumulative_2026.iloc[-1] - 1
) * 100

baseline_total_returns_2026

Static MVO      93.962868
Equal Weight    62.599968
Cash             0.000000
Name: 2026-09-15 00:00:00, dtype: float64

In [75]:
strategy_debug_2026 = pd.DataFrame({
    "Today_Momentum":
        market_state_2026["Momentum_20D"],

    "Signal_Momentum":
        signal_2026["Momentum_20D"],

    "Today_Leadership":
        market_state_2026["Leadership_Group"],

    "Signal_Leadership":
        signal_2026["Leadership_Group"],

    "Today_Gap":
        market_state_2026["Leadership_Gap_20D"],

    "Signal_Gap":
        signal_2026["Leadership_Gap_20D"],

    "Selected_Strategy":
        market_state_2026["Selected_Strategy_V1"]
})

strategy_debug_2026[
    strategy_debug_2026["Selected_Strategy"]
    == "EIIE 40ep"
]

,Today_Momentum,Signal_Momentum,Today_Leadership,Signal_Leadership,Today_Gap,Signal_Gap,Selected_Strategy
date,,,,,,,
2026-06-09,0.155271,0.139146,Semiconductor,NAVER,0.114725,0.189004,EIIE 40ep
2026-06-15,0.106376,0.084004,NAVER,NAVER,0.002704,0.090243,EIIE 40ep
2026-08-18,0.072382,0.042478,NAVER,NAVER,0.066884,0.123529,EIIE 40ep


In [76]:
static_mvo_return_2026 = (
    stock_returns_2026
    .mul(
        STATIC_MVO_WEIGHTS,
        axis=1
    )
    .sum(
        axis=1,
        min_count=1
    )
)

equal_weight_return_2026 = (
    stock_returns_2026
    .mul(
        EQUAL_WEIGHTS,
        axis=1
    )
    .sum(
        axis=1,
        min_count=1
    )
)

In [77]:
# EIIE 2026 inference

import torch

EIIE_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "policy_eiie_lr001_40ep.pt"
)

checkpoint = torch.load(
    EIIE_MODEL_PATH,
    map_location="cpu",
    weights_only=False
)

print(type(checkpoint))

<class 'collections.OrderedDict'>


In [78]:
if isinstance(checkpoint, dict):
    print(checkpoint.keys())

print(checkpoint)

odict_keys(['sequential.0.weight', 'sequential.0.bias', 'sequential.2.weight', 'sequential.2.bias', 'final_convolution.weight', 'final_convolution.bias'])
OrderedDict([('sequential.0.weight', tensor([[[[ 0.2665,  0.2897, -0.0604]],

         [[ 0.3210, -0.0526,  0.0873]],

         [[-0.1538,  0.2045,  0.3044]]],


        [[[-0.2354,  0.2923,  0.0657]],

         [[ 0.2504,  0.0502,  0.1597]],

         [[-0.0350,  0.2645,  0.0573]]]])), ('sequential.0.bias', tensor([-0.3540,  0.3102])), ('sequential.2.weight', tensor([[[[-0.0467, -0.0119, -0.0412,  ..., -0.0211,  0.0841, -0.0601]],

         [[-0.0605, -0.0605,  0.0912,  ..., -0.0946, -0.0666, -0.0338]]],


        [[[ 0.0159, -0.0893, -0.0437,  ...,  0.0223,  0.0131, -0.0894]],

         [[ 0.0426, -0.0152, -0.0465,  ...,  0.0343,  0.0647,  0.0468]]],


        [[[-0.0897, -0.0610, -0.0160,  ..., -0.0706,  0.0510,  0.0460]],

         [[ 0.0725, -0.0778,  0.0730,  ...,  0.0851, -0.0406,  0.0269]]],


        ...,


        [[[-0.036

In [79]:
#40ep EIIE 모델 복원

from finrl.agents.portfolio_optimization.architectures import EIIE

eiie_policy_2026 = EIIE(
    initial_features=3,
    k_size=3,
    conv_mid_features=2,
    conv_final_features=20,
    time_window=50,
    device="cpu"
)

load_result = eiie_policy_2026.load_state_dict(
    checkpoint
)

eiie_policy_2026.eval()

print(load_result)

C:\Users\SD1-06\miniforge3\envs\finrl\lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


<All keys matched successfully>


In [80]:
for name, param in eiie_policy_2026.named_parameters():
    print(
        name,
        tuple(param.shape)
    )

sequential.0.weight (2, 3, 1, 3)
sequential.0.bias (2,)
sequential.2.weight (20, 2, 1, 48)
sequential.2.bias (20,)
final_convolution.weight (1, 21, 1, 1)
final_convolution.bias (1,)


In [82]:
#train 데이터 다시 받아서 scaler 복원

from sklearn.preprocessing import MaxAbsScaler

FEATURES = [
    "close",
    "high",
    "low"
]

TICKER_ORDER = [
    "000660.KS",
    "005380.KS",
    "005930.KS",
    "035420.KS",
    "105560.KS"
]

In [83]:
#학습 구간

train_scaler_raw = YahooDownloader(
    start_date="2018-01-01",
    end_date="2024-01-01",
    ticker_list=TICKER_ORDER
).fetch_data()

train_scaler_raw["date"] = pd.to_datetime(
    train_scaler_raw["date"]
)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (7375, 8)


In [85]:
#공통 거래일만
train_date_counts = (
    train_scaler_raw
    .groupby("date")["tic"]
    .nunique()
)

train_common_dates = train_date_counts[
    train_date_counts == len(TICKER_ORDER)
].index

train_scaler_raw = (
    train_scaler_raw[
        train_scaler_raw["date"].isin(
            train_common_dates
        )
    ]
    .sort_values(
        ["date", "tic"]
    )
    .reset_index(drop=True)
)

print(
    train_scaler_raw
    .groupby("tic")
    .size()
)

tic
000660.KS    1475
005380.KS    1475
005930.KS    1475
035420.KS    1475
105560.KS    1475
dtype: int64


In [86]:
#종목별 train scaler

train_scalers = {}

for tic in TICKER_ORDER:

    tic_train = (
        train_scaler_raw[
            train_scaler_raw["tic"] == tic
        ]
        [FEATURES]
    )

    scaler = MaxAbsScaler()

    scaler.fit(
        tic_train
    )

    train_scalers[tic] = scaler

In [87]:
#scale값 확인

for tic, scaler in train_scalers.items():

    print(
        tic,
        scaler.scale_
    )

000660.KS [140775.609375   142671.60775303 137905.46875   ]
005380.KS [214747.78125    232007.88329439 208325.44414579]
005930.KS [81601.8046875  86802.79883242 80256.71999485]
035420.KS [441834.21875    452539.42097622 439887.78125   ]
105560.KS [53713.06640625 54202.85120631 52733.50608439]


In [88]:
#eiie용 2025말 + 2026 ohlc 데이터

eiie_inference_raw = YahooDownloader(
    start_date="2025-09-01",
    end_date="2026-09-16",
    ticker_list=TICKER_ORDER
).fetch_data()

eiie_inference_raw["date"] = pd.to_datetime(
    eiie_inference_raw["date"]
)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (1265, 8)


In [89]:
#공통 날짜

eiie_date_counts = (
    eiie_inference_raw
    .groupby("date")["tic"]
    .nunique()
)

eiie_common_dates = eiie_date_counts[
    eiie_date_counts == len(TICKER_ORDER)
].index

eiie_inference_raw = (
    eiie_inference_raw[
        eiie_inference_raw["date"].isin(
            eiie_common_dates
        )
    ]
    .sort_values(
        ["date", "tic"]
    )
    .reset_index(drop=True)
)

In [90]:
#원본 보존 copy

eiie_scaled = (
    eiie_inference_raw
    .copy()
)

In [91]:
for tic in TICKER_ORDER:

    mask = (
        eiie_scaled["tic"]
        == tic
    )

    eiie_scaled.loc[
        mask,
        FEATURES
    ] = train_scalers[tic].transform(
        eiie_scaled.loc[
            mask,
            FEATURES
        ]
    )

In [92]:
#확인

eiie_scaled[
    [
        "date",
        "tic",
        "close",
        "high",
        "low"
    ]
].head(10)

Price,date,tic,close,high,low
0,2025-09-01,000660.KS,1.813152,1.823999,1.843658
1,2025-09-01,005380.KS,1.002471,0.938413,1.026346
2,2025-09-01,005930.KS,0.818146,0.780503,0.830627
3,2025-09-01,035420.KS,0.482669,0.473438,0.473557
4,2025-09-01,105560.KS,1.933928,1.957609,1.968013
5,2025-09-02,000660.KS,1.845024,1.827493,1.850888
6,2025-09-02,005380.KS,1.000198,0.929997,1.016973
7,2025-09-02,005930.KS,0.836300,0.790742,0.834319
8,2025-09-02,035420.KS,0.503947,0.500773,0.485930
9,2025-09-02,105560.KS,1.964625,1.949557,1.960656


In [93]:
#EIIE observation: feature x stock x time (3x5x50)

#각 feature pivot

close_matrix = (
    eiie_scaled
    .pivot(
        index="date",
        columns="tic",
        values="close"
    )
    .reindex(
        columns=TICKER_ORDER
    )
)

high_matrix = (
    eiie_scaled
    .pivot(
        index="date",
        columns="tic",
        values="high"
    )
    .reindex(
        columns=TICKER_ORDER
    )
)

low_matrix = (
    eiie_scaled
    .pivot(
        index="date",
        columns="tic",
        values="low"
    )
    .reindex(
        columns=TICKER_ORDER
    )
)

In [94]:
#확인
print(close_matrix.shape)

print(
    close_matrix.index.min(),
    close_matrix.index.max()
)

print(close_matrix.columns.tolist())

(253, 5)
2025-09-01 00:00:00 2026-09-15 00:00:00
['000660.KS', '005380.KS', '005930.KS', '035420.KS', '105560.KS']


In [95]:
print(load_result)

print(
    train_scaler_raw
    .groupby("tic")
    .size()
)

print(
    close_matrix.shape
)

print(
    close_matrix.index.min(),
    close_matrix.index.max()
)

<All keys matched successfully>
tic
000660.KS    1475
005380.KS    1475
005930.KS    1475
035420.KS    1475
105560.KS    1475
dtype: int64
(253, 5)
2025-09-01 00:00:00 2026-09-15 00:00:00


In [96]:
TIME_WINDOW = 50

ACTION_COLUMNS = [
    "Cash",
    "000660.KS",
    "005380.KS",
    "005930.KS",
    "035420.KS",
    "105560.KS"
]

all_dates = close_matrix.index

# FinRL 환경과 동일하게 처음에는 100% 현금
last_action = np.array(
    [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]],
    dtype=np.float32
)

eiie_action_records = []

eiie_policy_2026.eval()

with torch.no_grad():

    # i = 현재 observation window의 마지막 날짜
    # 마지막 날짜에는 다음 거래일이 없으므로 len - 1까지만
    for i in range(
        TIME_WINDOW - 1,
        len(all_dates) - 1
    ):

        signal_date = all_dates[i]
        trade_date = all_dates[i + 1]

        start_idx = i - TIME_WINDOW + 1
        end_idx = i + 1

        # 각각 shape = (5 stocks, 50 days)
        close_window = (
            close_matrix
            .iloc[start_idx:end_idx]
            .T
            .to_numpy()
        )

        high_window = (
            high_matrix
            .iloc[start_idx:end_idx]
            .T
            .to_numpy()
        )

        low_window = (
            low_matrix
            .iloc[start_idx:end_idx]
            .T
            .to_numpy()
        )

        # shape:
        # (3 features, 5 stocks, 50 days)
        observation = np.stack(
            [
                close_window,
                high_window,
                low_window
            ],
            axis=0
        )

        # batch dimension 추가
        # (1, 3, 5, 50)
        observation = (
            observation
            .astype(np.float32)
            [np.newaxis, ...]
        )

        # EIIE action
        action = eiie_policy_2026(
            observation,
            last_action
        )

        action = np.asarray(
            action,
            dtype=np.float32
        ).reshape(-1)

        eiie_action_records.append(
            {
                "Signal_Date": signal_date,
                "Trade_Date": trade_date,
                **{
                    col: value
                    for col, value
                    in zip(
                        ACTION_COLUMNS,
                        action
                    )
                }
            }
        )

        # 다음 EIIE inference에 사용
        last_action = action.reshape(1, -1)

In [97]:
#dataframe으로 만들기

eiie_weights_all = pd.DataFrame(
    eiie_action_records
)

eiie_weights_all[
    "Signal_Date"
] = pd.to_datetime(
    eiie_weights_all["Signal_Date"]
)

eiie_weights_all[
    "Trade_Date"
] = pd.to_datetime(
    eiie_weights_all["Trade_Date"]
)

eiie_weights_all = (
    eiie_weights_all
    .set_index("Trade_Date")
)

eiie_weights_all.head()

,Signal_Date,Cash,000660.KS,005380.KS,005930.KS,035420.KS,105560.KS
Trade_Date,,,,,,,
2025-11-18,2025-11-17,0.193755,2.001415e-11,0.002008,0.001486,0.802751,1.143261e-07
2025-11-19,2025-11-18,0.156428,1.038721e-11,0.001534,0.001099,0.840940,8.811746e-08
2025-11-20,2025-11-19,0.156425,6.824969e-12,0.001474,0.001032,0.841069,8.430510e-08
2025-11-21,2025-11-20,0.158427,4.716086e-12,0.001410,0.000975,0.839189,8.254555e-08
2025-11-24,2025-11-21,0.160146,3.342878e-12,0.001335,0.000915,0.837604,8.194066e-08


In [98]:
#2026만

eiie_weights_2026 = (
    eiie_weights_all
    .loc["2026-01-02":"2026-09-15"]
    .copy()
)

print(
    eiie_weights_2026.index.min(),
    eiie_weights_2026.index.max()
)

print(
    eiie_weights_2026.shape
)

eiie_weights_2026.head()

2026-01-02 00:00:00 2026-09-15 00:00:00
(173, 7)


,Signal_Date,Cash,000660.KS,005380.KS,005930.KS,035420.KS,105560.KS
Trade_Date,,,,,,,
2026-01-02,2025-12-30,0.143910,1.641158e-14,0.000231,0.000244,0.855615,2.511416e-08
2026-01-05,2026-01-02,0.141517,1.451595e-14,0.000217,0.000231,0.858034,2.417802e-08
2026-01-06,2026-01-05,0.138657,1.251004e-14,0.000200,0.000215,0.860928,2.351377e-08
2026-01-07,2026-01-06,0.136694,1.044560e-14,0.000187,0.000192,0.862927,2.265673e-08
2026-01-08,2026-01-07,0.135048,8.378166e-15,0.000178,0.000174,0.864599,2.183516e-08


In [99]:
#비중이 정상인지 검사

weight_columns = ACTION_COLUMNS

weight_sums = (
    eiie_weights_2026[
        weight_columns
    ]
    .sum(axis=1)
)

print(
    "Weight sum min:",
    weight_sums.min()
)

print(
    "Weight sum max:",
    weight_sums.max()
)

print(
    "Minimum weight:",
    eiie_weights_2026[
        weight_columns
    ]
    .min()
    .min()
)

Weight sum min: 0.9999999403953552
Weight sum max: 1.0000001192092896
Minimum weight: 0.0


In [100]:
#2026 평균 비중 확인

eiie_avg_weights_2026 = (
    eiie_weights_2026[
        weight_columns
    ]
    .mean()
)

eiie_avg_weights_2026

Cash         9.232773e-02
000660.KS    5.633768e-16
005380.KS    1.290214e-05
005930.KS    1.304531e-05
035420.KS    9.076463e-01
105560.KS    3.620181e-09
dtype: float32

In [101]:
#최대 비중 종목
eiie_weights_2026[
    weight_columns
].idxmax(axis=1).value_counts()

035420.KS    173
Name: count, dtype: int64